# Loading the Libraries for the Prediction Model 

In [37]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from xgboost import XGBClassifier, XGBRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, log_loss, mean_absolute_error
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

# Web Scraping

In [ ]:
URL = "https://www.transfermarkt.com/weltmeisterschaft/teilnehmer/pokalwettbewerb/FIWC/saison_id/2025"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}

response = requests.get(URL, headers=HEADERS)
response.raise_for_status()

soup = BeautifulSoup(response.content, "html.parser")

teams = []

for table in soup.select("table.items"):
    header_cells = [th.get_text(strip=True) for th in table.select("thead th")]
    header_cells = [h for h in header_cells if h]

    for tr in table.select("tbody tr"):
        cells = tr.find_all("td")
        if not cells:
            continue

        club_link = tr.find("a", href=lambda h: h and "/startseite/verein/" in h)
        if not club_link:
            continue

        texts = [td.get_text(strip=True) for td in cells if td.get_text(strip=True)]

        row = {
            "Club": club_link.get_text(strip=True),
        }

        n = min(len(header_cells), len(texts))
        for h, v in zip(header_cells[-n:], texts[-n:]):
            row[h] = v

        teams.append(row)

df = pd.DataFrame(teams)
print(df)
df.to_csv("world_cup_2026_participants.csv", index=False)

# Loading the Data and Cleaning

In [38]:
Elo = pd.read_csv("C:\\Project DATA\\2026 Fifa WC Winner\\Data\\elo_ratings_wc2026.csv")
WC_2026_Squads = pd.read_csv("Data\\world_cup_2026_participants.csv",  encoding='latin-1')
Results = pd.read_csv("C:\\Project DATA\\2026 Fifa WC Winner\\Data\\results.csv")
Shootouts = pd.read_csv("C:\\Project DATA\\2026 Fifa WC Winner\\Data\\shootouts.csv")
Schedule = pd.read_csv("C:\\Project DATA\\2026 Fifa WC Winner\\Data\\world-cup-2026-schedule.csv")

In [39]:
team_dictionary = {
    'Czech Republic': 'Czechia',                              
    'Bosnia-Herzegovina': 'Bosnia and Herzegovina',            
    'Democratic Republic of the Congo': 'DR Congo',            
    'Turkiye': 'Turkey',                                       
    'Cabo Verde': 'Cape Verde',                                
    'Congo DR': 'DR Congo',                                   
    'Côte d’Ivoire': 'Ivory Coast',                            
    'Korea Republic': 'South Korea',                           
    'Türkiye': 'Turkey',    
    'United States': 'USA',   
    'Congo DR' : 'DR Congo',
    "Curaçao": "Curacao", 
    "CuraÃ§ao": "Curacao",                           
}

In [40]:
# Grabbing the first 48 rows of the squads data to ensure we have only the participating teams.
WC_2026_Squads_Cleaned = WC_2026_Squads.iloc[0:48].copy()

WC_2026_Squads_Cleaned['Club'] = WC_2026_Squads_Cleaned['Club'].str.strip().replace(team_dictionary)

WC_2026_Squads_Cleaned = WC_2026_Squads_Cleaned.rename(columns={
    'Club': 'team',
    '&oslash-Age': 'Average Age',
    '&oslash-Market Value': 'Average Market Value(In Euros)',
    'Market Value': 'Market Value(In Euros)',
    'WC particip.': 'WC Participations',
})

WC_2026_Squads_Cleaned['Foreigners'] = (
    WC_2026_Squads_Cleaned['Foreigners'].str.rstrip(' %').astype(float) / 100
)

def parse_value(x):
    x = x.replace('â¬', '').strip()
    if x.endswith('bn'):
        return float(x[:-2]) * 1_000_000_000
    if x.endswith('m'):
        return float(x[:-1]) * 1_000_000
    if x.endswith('k'):
        return float(x[:-1]) * 1_000
    return float(x)

WC_2026_Squads_Cleaned['Market Value(In Euros)'] = WC_2026_Squads_Cleaned['Market Value(In Euros)'].apply(parse_value)
WC_2026_Squads_Cleaned['Average Market Value(In Euros)'] = WC_2026_Squads_Cleaned['Average Market Value(In Euros)'].apply(parse_value)

print(WC_2026_Squads_Cleaned.head())

       team  Squad  Average Age  WC Participations  Foreigners  \
0    France     26         27.0                 17       0.731   
1   England     26         27.2                 17       0.192   
2     Spain     26         26.8                 18       0.346   
3  Portugal     26         28.1                 10       0.808   
4   Germany     26         28.1                 21       0.269   

   Market Value(In Euros)  Average Market Value(In Euros)  
0            1.520000e+09                      58580000.0  
1            1.360000e+09                      52430000.0  
2            1.220000e+09                      47030000.0  
3            1.010000e+09                      38670000.0  
4            9.470000e+08                      36420000.0  


In [41]:
Results['date'] = pd.to_datetime(Results['date'])

Results['home_team'] = Results['home_team'].str.strip().replace(team_dictionary)
Results['away_team'] = Results['away_team'].str.strip().replace(team_dictionary)

Results = Results[Results['home_score'].notna()].copy()
Results['home_score'] = Results['home_score'].astype(int)
Results['away_score'] = Results['away_score'].astype(int)

print(Results.head())

        date home_team away_team  home_score  away_score tournament     city  \
0 1872-11-30  Scotland   England           0           0   Friendly  Glasgow   
1 1873-03-08   England  Scotland           4           2   Friendly   London   
2 1874-03-07  Scotland   England           2           1   Friendly  Glasgow   
3 1875-03-06   England  Scotland           2           2   Friendly   London   
4 1876-03-04  Scotland   England           3           0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  


In [42]:
Schedule['date'] = pd.to_datetime(Schedule['date'])

Schedule['team_a'] = Schedule['team_a'].str.strip().replace(team_dictionary)
Schedule['team_b'] = Schedule['team_b'].str.strip().replace(team_dictionary)
Schedule= Schedule.drop(columns=['source'], axis=1)

print(Schedule.head())

   match_number        stage group       date time_et time_local       team_a  \
0             1  Group Stage     A 2026-06-11   15:00      13:00       Mexico   
1             2  Group Stage     A 2026-06-11   22:00      20:00  South Korea   
2             3  Group Stage     B 2026-06-12   15:00      15:00       Canada   
3             4  Group Stage     D 2026-06-12   21:00      18:00          USA   
4             5  Group Stage     C 2026-06-13   21:00      21:00        Haiti   

                   team_b             venue         city        country  \
0            South Africa    Estadio Azteca  Mexico City         Mexico   
1                 Czechia     Estadio Akron  Guadalajara         Mexico   
2  Bosnia and Herzegovina         BMO Field      Toronto         Canada   
3                Paraguay      SoFi Stadium  Los Angeles  United States   
4                Scotland  Gillette Stadium       Boston  United States   

                    status  
0  confirmed_group_fixture  
1  c

In [43]:
Shootouts['date'] = pd.to_datetime(Shootouts['date'])

Shootouts['home_team'] = Shootouts['home_team'].str.strip().replace(team_dictionary)
Shootouts['away_team'] = Shootouts['away_team'].str.strip().replace(team_dictionary)
Shootouts['winner'] = Shootouts['winner'].str.strip().replace(team_dictionary)

print(Shootouts.head())

        date    home_team         away_team       winner first_shooter
0 1967-08-22        India            Taiwan       Taiwan           NaN
1 1971-11-14  South Korea  Vietnam Republic  South Korea           NaN
2 1972-05-07  South Korea              Iraq         Iraq           NaN
3 1972-05-17     Thailand       South Korea  South Korea           NaN
4 1972-05-19     Thailand          Cambodia     Thailand           NaN


In [44]:
Elo['snapshot_date'] = pd.to_datetime(Elo['snapshot_date'])
Elo['country'] = Elo['country'].str.strip().replace(team_dictionary)

Elo_Pre_WC = Elo[Elo['snapshot_date'] == '2026-05-27'].copy()

# Feature Engineering

In [45]:
WC2026_Nations = set(Elo['country'].unique())

Results_filter = Results[
    Results['home_team'].isin(WC2026_Nations) &
    Results['away_team'].isin(WC2026_Nations)
].copy()

Elo_Filter = (
    Elo[['country', 'snapshot_date', 'rating']]
    .sort_values('snapshot_date')
    .reset_index(drop=True)
)

Results_sorted = Results_filter.sort_values('date').reset_index(drop=True)

Results_with_elo = pd.merge_asof(
    Results_sorted,
    Elo_Filter.rename(columns={'country': 'home_team', 'rating': 'home_elo', 'snapshot_date': 'home_snap'}),
    left_on='date',
    right_on='home_snap',
    by='home_team',
    direction='backward'
).drop(columns='home_snap')

Results_with_elo = pd.merge_asof(
    Results_with_elo,
    Elo_Filter.rename(columns={'country': 'away_team', 'rating': 'away_elo', 'snapshot_date': 'away_snap'}),
    left_on='date',
    right_on='away_snap',
    by='away_team',
    direction='backward'
).drop(columns='away_snap')

Results_with_elo['elo_diff'] = Results_with_elo['home_elo'] - Results_with_elo['away_elo']

Results_with_elo = Results_with_elo.dropna(subset=['home_elo', 'away_elo'])
Results_with_elo['date'] = pd.to_datetime(Results_with_elo['date'])

Results_Elo = Results_with_elo[(Results_with_elo['date'].dt.year >= 1990)]

In [46]:
Home_View = Results_Elo[['date','home_team','away_team','home_score','away_score']].copy()
Home_View.columns = ['date','team','opponent','gf','ga']

Away_View = Results_Elo[['date','away_team','home_team','away_score','home_score']].copy()
Away_View.columns = ['date','team','opponent','gf','ga']

Team_Games = pd.concat([Home_View, Away_View], ignore_index=True)
Team_Games['win'] = (Team_Games['gf'] > Team_Games['ga']).astype(int)
Team_Games = Team_Games.sort_values(['team','date']).reset_index(drop=True)

Team_Games['form_gf'] = Team_Games.groupby('team')['gf'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
Team_Games['form_ga'] = Team_Games.groupby('team')['ga'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
Team_Games['form_win_rate'] = Team_Games.groupby('team')['win'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())

form = Team_Games[['date','team','form_gf','form_ga','form_win_rate']]

Results_Featured = Results_Elo.merge(
    form.rename(columns={'team':'home_team','form_gf':'home_form_gf','form_ga':'home_form_ga','form_win_rate':'home_form_win_rate'}),
    on=['date','home_team'], how='left'
)

Results_Featured = Results_Featured.merge(
    form.rename(columns={'team':'away_team','form_gf':'away_form_gf','form_ga':'away_form_ga','form_win_rate':'away_form_win_rate'}),
    on=['date','away_team'], how='left'
)

print(Results_Featured[['date','home_team','away_team','home_form_gf','away_form_win_rate']].tail())

           date    home_team    away_team  home_form_gf  away_form_win_rate
4234 2026-06-01      Austria      Tunisia           2.8                 0.6
4235 2026-06-01       Norway       Sweden           1.0                 0.2
4236 2026-06-02        Haiti  New Zealand           0.6                 0.0
4237 2026-06-02      Croatia      Belgium           1.6                 0.2
4238 2026-06-03  Netherlands      Algeria           1.8                 0.6


In [47]:
H2H_Home = Results_Elo[['date','home_team','away_team','home_score','away_score']].copy()
H2H_Home['team'] = H2H_Home['home_team']
H2H_Home['opponent'] = H2H_Home['away_team']
H2H_Home['won'] = (H2H_Home['home_score'] > H2H_Home['away_score']).astype(int)

H2H_Away = Results_Elo[['date','home_team','away_team','home_score','away_score']].copy()
H2H_Away['team'] = H2H_Away['away_team']
H2H_Away['opponent'] = H2H_Away['home_team']
H2H_Away['won'] = (H2H_Away['away_score'] > H2H_Away['home_score']).astype(int)

H2H = (
    pd.concat([H2H_Home, H2H_Away], ignore_index=True)
    [['date','team','opponent','won']]
    .sort_values(['team','opponent','date'])
    .reset_index(drop=True)
)

H2H['h2h_cum_wins'] = H2H.groupby(['team','opponent'])['won'].transform(lambda x: x.shift(1).expanding().sum())
H2H['h2h_cum_games'] = H2H.groupby(['team','opponent'])['won'].transform(lambda x: x.shift(1).expanding().count())
H2H['h2h_win_rate'] = H2H['h2h_cum_wins'] / H2H['h2h_cum_games']  

Results_Featured = Results_Featured.merge(
    H2H[['date','team','opponent','h2h_win_rate','h2h_cum_games']]
      .rename(columns={'team':'home_team','opponent':'away_team',
                       'h2h_win_rate':'home_h2h_win_rate','h2h_cum_games':'h2h_total_games'}),
    on=['date','home_team','away_team'], how='left'
)

In [ ]:
def result_game(row):
    if row['home_score'] > row['away_score']:return 'win'
    elif row['home_score'] < row['away_score']:return 'loss'
    else:return 'draw'

Results_Featured['outcome'] = Results_Featured.apply(result_game, axis=1)

           date    home_team    away_team  home_score  away_score outcome
4234 2026-06-01      Austria      Tunisia           1           0     win
4235 2026-06-01       Norway       Sweden           3           1     win
4236 2026-06-02        Haiti  New Zealand           4           0     win
4237 2026-06-02      Croatia      Belgium           0           2    loss
4238 2026-06-03  Netherlands      Algeria           0           1    loss


In [49]:
Results_Featured['home_h2h_win_rate'] = Results_Featured['home_h2h_win_rate'].fillna(0.5)
Results_Featured = Results_Featured.dropna(subset=['home_form_gf', 'away_form_gf'])

In [50]:
Results_Featured['WC'] = (Results_Featured['tournament'] == 'FIFA World Cup').astype(int)
Results_Featured['Qualifier'] = (Results_Featured['tournament'] == 'FIFA World Cup qualification').astype(int)

Results_Featured.to_csv("C:\\Project DATA\\2026 Fifa WC Winner\\Data\\results_featured.csv", index=False)

In [60]:
Shootout_Home = Shootouts[['home_team','winner']].rename(columns={'home_team':'team'})
Shootout_Home['won'] = (Shootout_Home['winner'] == Shootout_Home['team']).astype(int)

Shootout_Away = Shootouts[['away_team','winner']].rename(columns={'away_team':'team'})
Shootout_Away['won'] = (Shootout_Away['winner'] == Shootout_Away['team']).astype(int)

Shootout_Rates = (
    pd.concat([Shootout_Home, Shootout_Away], ignore_index=True)
    .groupby('team')['won']
    .agg(shootout_wins='sum', shootout_games='count')
    .reset_index()
)
Shootout_Rates['shootout_win_rate'] = Shootout_Rates['shootout_wins'] / Shootout_Rates['shootout_games']

print(Shootout_Rates.sort_values('shootout_win_rate', ascending=False).head())

                   team  shootout_wins  shootout_games  shootout_win_rate
13              Bahrain              3               3                1.0
5              Anguilla              1               1                1.0
6   Antigua and Barbuda              2               2                1.0
16       Basque Country              1               1                1.0
18              Belgium              2               2                1.0


In [ ]:
Squad_Features = WC_2026_Squads_Cleaned[['team','Squad','Average Age','WC Participations',
                                          'Foreigners','Market Value(In Euros)','Average Market Value(In Euros)']]

Schedule_Squad_Info = (
    Schedule
    .merge(
        Squad_Features.add_prefix('a_').rename(columns={'a_team':'team_a'}),
        on='team_a', how='left'
    )
    .merge(
        Squad_Features.add_prefix('b_').rename(columns={'b_team':'team_b'}),
        on='team_b', how='left'
    )
)

        team_a                  team_b  a_Market Value(In Euros)  \
0       Mexico            South Africa               191850000.0   
1  South Korea                 Czechia               139050000.0   
2       Canada  Bosnia and Herzegovina               198650000.0   
3          USA                Paraguay               385650000.0   
4        Haiti                Scotland                55900000.0   
5    Australia                  Turkey                77450000.0   
6       Brazil                 Morocco               928200000.0   
7        Qatar             Switzerland                19930000.0   
8  Ivory Coast                 Ecuador               522100000.0   
9      Germany                 Curacao               947000000.0   

   b_Market Value(In Euros)  
0                49250000.0  
1               188180000.0  
2               146400000.0  
3               153650000.0  
4               170250000.0  
5               473700000.0  
6               447700000.0  
7          

# Training and Testing

In [53]:
LE = LabelEncoder()
Results_Featured['outcome_encoded'] = LE.fit_transform(Results_Featured['outcome'])

Results_Featured = Results_Featured.sort_values('date').reset_index(drop=True)

train = Results_Featured[Results_Featured['date'] < '2018-01-01']
val = Results_Featured[(Results_Featured['date'] >= '2018-01-01') & (Results_Featured['date'] < '2022-11-20')]
test = Results_Featured[Results_Featured['date'] >= '2022-11-20']

In [54]:
FEATURES = [
    'home_elo', 'away_elo', 'elo_diff',
    'home_form_gf', 'home_form_ga', 'home_form_win_rate',
    'away_form_gf', 'away_form_ga', 'away_form_win_rate',
    'home_h2h_win_rate', 'h2h_total_games',
    'neutral','WC', "Qualifier"
]

X_train = train[FEATURES]
X_val   = val[FEATURES]
X_test  = test[FEATURES]

y_train = train['outcome_encoded']
y_val   = val['outcome_encoded']
y_test  = test['outcome_encoded']

model_a = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    n_estimators=800,
    learning_rate=0.01,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=20,
    random_state=42,
    min_child_weight=2,
)

model_a.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

# Evaluate
val_probs  = model_a.predict_proba(X_val)
test_probs = model_a.predict_proba(X_test)

print(f"\nVal  log loss: {log_loss(y_val, val_probs):.4f}")
print(f"Test log loss: {log_loss(y_test, test_probs):.4f}")
print(f"Val  accuracy: {accuracy_score(y_val, val_probs.argmax(axis=1)):.4f}")
print(f"Test accuracy: {accuracy_score(y_test, test_probs.argmax(axis=1)):.4f}")


[0]	validation_0-mlogloss:1.09692
[50]	validation_0-mlogloss:1.04285
[100]	validation_0-mlogloss:1.01640
[150]	validation_0-mlogloss:1.00267
[200]	validation_0-mlogloss:0.99590
[250]	validation_0-mlogloss:0.99235
[300]	validation_0-mlogloss:0.99096
[350]	validation_0-mlogloss:0.99026
[368]	validation_0-mlogloss:0.99028

Val  log loss: 0.9902
Test log loss: 1.0283
Val  accuracy: 0.5182
Test accuracy: 0.4645


In [55]:
model_b_home = XGBRegressor(
    objective='count:poisson',   
    eval_metric='mae',
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=20,
    random_state=42,
    min_child_weight=2
)

model_b_home.fit(
    X_train, train['home_score'],
    eval_set=[(X_val, val['home_score'])],
    verbose=50
)

model_b_away = XGBRegressor(
    objective='count:poisson',
    eval_metric='mae',
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=20,
    random_state=42,
    min_child_weight=2
)

model_b_away.fit(
    X_train, train['away_score'],
    eval_set=[(X_val, val['away_score'])],
    verbose=50
)

print(f"\nHome goals MAE — val:  {mean_absolute_error(val['home_score'],  model_b_home.predict(X_val)):.4f}")
print(f"Home goals MAE — test: {mean_absolute_error(test['home_score'], model_b_home.predict(X_test)):.4f}")
print(f"Away goals MAE — val:  {mean_absolute_error(val['away_score'],  model_b_away.predict(X_val)):.4f}")
print(f"Away goals MAE — test: {mean_absolute_error(test['away_score'], model_b_away.predict(X_test)):.4f}")

[0]	validation_0-mae:1.06780


[50]	validation_0-mae:1.01174
[100]	validation_0-mae:1.00297
[128]	validation_0-mae:1.00332
[0]	validation_0-mae:0.83513
[36]	validation_0-mae:0.82496

Home goals MAE — val:  1.0022
Home goals MAE — test: 1.0099
Away goals MAE — val:  0.8242
Away goals MAE — test: 0.8614


In [ ]:
test_probs = model_a.predict_proba(X_test)  

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, outcome_label in enumerate(LE.classes_):
    y_true_binary = (y_test == i).astype(int)
    prob_true, prob_pred = calibration_curve(y_true_binary, test_probs[:, i], n_bins=10)
    
    axes[i].plot(prob_pred, prob_true, marker='o', label='Model A')
    axes[i].plot([0,1], [0,1], linestyle='--', color='grey', label='Perfect')
    axes[i].set_title(f'Calibration — {outcome_label}')
    axes[i].set_xlabel('Predicted probability')
    axes[i].set_ylabel('Actual frequency')
    axes[i].legend()

plt.show()

# Poisson Match Simulation

In [ ]:
def simulate_match(home_team, away_team, feature_row, model_b_home, model_b_away,
                   shootout_rates, knockout=False, n_sims=10_000):
    
    lambda_home = model_b_home.predict(feature_row)[0]
    lambda_away = model_b_away.predict(feature_row)[0]
    
    home_goals = np.random.poisson(lambda_home, n_sims)
    away_goals = np.random.poisson(lambda_away, n_sims)
    
    home_wins = (home_goals > away_goals).sum()
    away_wins = (away_goals > home_goals).sum()
    draws      = (home_goals == away_goals).sum()
    
    results = {
        'home_win_pct': home_wins / n_sims,
        'away_win_pct': away_wins / n_sims,
        'draw_pct':     draws / n_sims,
        'lambda_home':  lambda_home,
        'lambda_away':  lambda_away,
    }
    
    if knockout:
        et_lambda_home = lambda_home * 0.6 * (30/90)  
        et_lambda_away = lambda_away * 0.6 * (30/90)
        
        draw_indices = np.where(home_goals == away_goals)[0]
        
        et_home = np.random.poisson(et_lambda_home, len(draw_indices))
        et_away = np.random.poisson(et_lambda_away, len(draw_indices))
        
        et_home_wins = (et_home > et_away).sum()
        et_away_wins = (et_away > et_home).sum()
        et_still_draw = (et_home == et_away).sum()
        
        home_pen_rate = shootout_rates.set_index('team')['shootout_win_rate'].get(home_team, 0.5)
        away_pen_rate = shootout_rates.set_index('team')['shootout_win_rate'].get(away_team, 0.5)
        
        total = home_pen_rate + away_pen_rate
        home_pen_rate /= total
        
        pen_home_wins = np.random.binomial(et_still_draw, home_pen_rate)
        pen_away_wins = et_still_draw - pen_home_wins
        
        results['knockout_home_win_pct'] = (home_wins + et_home_wins + pen_home_wins) / n_sims
        results['knockout_away_win_pct'] = (away_wins + et_away_wins + pen_away_wins) / n_sims
    
    return results

# Monte Carlo Simulation and Result